# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [19]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib
from sklearn.pipeline import Pipeline

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [20]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        X_copy['timestamp'] = pd.to_datetime(X_copy['timestamp'])
        X_copy['hour'] = X_copy['timestamp'].dt.hour
        # X_copy['weekday'] = X_copy['timestamp'].dt.weekday
        X_copy = X_copy.drop('timestamp', axis=1)
        return X_copy

In [21]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, target_column):
        self.target_column = target_column
        self.encoder = None
        self.categorical_cols = None
        
    def fit(self, X, y=None):
        self.categorical_cols = [
            col for col in X.columns 
            if X[col].dtype == 'object' and col != self.target_column
        ]
        self.encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')
        self.encoder.fit(X[self.categorical_cols])
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        if self.categorical_cols:
            encoded = self.encoder.transform(X_copy[self.categorical_cols])
            encoded_df = pd.DataFrame(
                encoded, 
                columns=self.encoder.get_feature_names(self.categorical_cols),
                index=X_copy.index
            )
            X_copy = X_copy.drop(self.categorical_cols, axis=1)
            X_copy = pd.concat([X_copy, encoded_df], axis=1)
        if self.target_column in X_copy.columns:
            y = X_copy[self.target_column]
            X_copy = X_copy.drop(self.target_column, axis=1)
            return X_copy, y
        return X_copy

In [22]:
class TrainValidationTest(BaseEstimator, TransformerMixin):
    def __init__(self, test_size=0.2, valid_size=0.2, random_state=21):
        self.test_size = test_size
        self.valid_size = valid_size
        self.random_state = random_state
        self.y_ = None
        
    def fit(self, X, y=None):
        self.y_ = y
        return self
    
    def transform(self, X, y=None):
        X_train_full, X_test, y_train_full, y_test = train_test_split(
            X, self.y_, test_size=self.test_size, random_state=self.random_state, stratify=self.y_
        )
        X_train, X_valid, y_train, y_valid = train_test_split(
            X_train_full, y_train_full, test_size=self.valid_size, 
            random_state=self.random_state, stratify=y_train_full
        )
        return X_train, X_valid, X_test, y_train, y_valid, y_test

In [23]:
# Загрузка данных
df_raw = pd.read_csv('../data/checker_submits.csv')
df_target = pd.read_csv('../data/dayofweek.csv')
df = df_raw.copy()
df['dayofweek'] = df_target['dayofweek']

# Шаг 1: FeatureExtractor
fe = FeatureExtractor()
df_extracted = fe.fit_transform(df)

# Шаг 2: MyOneHotEncoder
moe = MyOneHotEncoder(target_column='dayofweek')
X, y = moe.fit_transform(df_extracted)

# Шаг 3: TrainValidationTest
tvt = TrainValidationTest(test_size=0.2, valid_size=0.2, random_state=21)
X_train, X_valid, X_test, y_train, y_valid, y_test = tvt.fit_transform(X, y)

# Вывод результатов
print(f'X_train: {X_train.shape}')
print(f'X_valid: {X_valid.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}')

X_train: (1078, 43)
X_valid: (270, 43)
X_test: (338, 43)
y_train: (1078,)


## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [24]:
class ModelSelection:
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.best_models = {}
        self.results = []
        
    def choose(self, X_train, y_train, X_valid, y_valid):
        for i, grid in enumerate(tqdm(self.grids, desc='Models')):
            model_name = self.grid_dict[i]
            print(f'\nEstimator: {model_name}')
            
            grid.fit(X_train, y_train)
            
            best_params = grid.best_params_
            best_model = grid.best_estimator_
            
            train_pred = best_model.predict(X_train)
            valid_pred = best_model.predict(X_valid)
            
            train_acc = accuracy_score(y_train, train_pred)
            valid_acc = accuracy_score(y_valid, valid_pred)
            
            print(f'Best params: {best_params}')
            print(f'Best training accuracy: {train_acc:.3f}')
            print(f'Validation set accuracy score for best params: {valid_acc:.3f}')
            
            self.best_models[model_name] = {
                'model': best_model,
                'params': best_params,
                'valid_score': valid_acc
            }
            
            self.results.append({
                'model': model_name,
                'params': best_params,
                'valid_score': valid_acc
            })
        
        best_model_name = max(self.best_models, key=lambda x: self.best_models[x]['valid_score'])
        print(f'\nClassifier with best validation set accuracy: {best_model_name}')
        
        return best_model_name
    
    def best_results(self):
        return pd.DataFrame(self.results)

In [25]:
jobs = -1

svm_params = [{
    'kernel': ['linear', 'rbf', 'sigmoid'],
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced', None],
    'random_state': [21],
    'probability': [True]
}]

tree_params = [{
    'max_depth': [5, 10, 15, 20, 21, 25, 30],
    'criterion': ['gini', 'entropy'],
    'class_weight': ['balanced', None],
    'random_state': [21]
}]

rf_params = [{
    'n_estimators': [10, 30, 50, 70, 100],
    'max_depth': [10, 15, 20, 22, 24, 30],
    'criterion': ['gini', 'entropy'],
    'class_weight': ['balanced', None],
    'random_state': [21]
}]

gs_svm = GridSearchCV(estimator=SVC(), param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs)
gs_tree = GridSearchCV(estimator=DecisionTreeClassifier(), param_grid=tree_params, scoring='accuracy', cv=2, n_jobs=jobs)
gs_rf = GridSearchCV(estimator=RandomForestClassifier(), param_grid=rf_params, scoring='accuracy', cv=2, n_jobs=jobs)

grids = [gs_svm, gs_tree, gs_rf]
grid_dict = {0: 'SVM', 1: 'Decision Tree', 2: 'Random Forest'}

In [26]:
ms = ModelSelection(grids, grid_dict)
best_classifier = ms.choose(X_train, y_train, X_valid, y_valid)

Models:   0%|          | 0/3 [00:00<?, ?it/s]


Estimator: SVM


Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.952
Validation set accuracy score for best params: 0.878

Estimator: Decision Tree
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.994
Validation set accuracy score for best params: 0.867

Estimator: Random Forest
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 20, 'n_estimators': 70, 'random_state': 21}
Best training accuracy: 1.000
Validation set accuracy score for best params: 0.904

Classifier with best validation set accuracy: Random Forest


In [27]:
results_df = ms.best_results()
print(results_df.to_string(index=False))

         model                                                                                                      params  valid_score
           SVM  {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}     0.877778
 Decision Tree                      {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}     0.866667
 Random Forest     {'class_weight': None, 'criterion': 'entropy', 'max_depth': 20, 'n_estimators': 70, 'random_state': 21}     0.903704


## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [28]:
class Finalize:
    def __init__(self, estimator):
        self.estimator = estimator
    
    def final_score(self, X_train, y_train, X_test, y_test):
        self.estimator.fit(X_train, y_train)
        score = self.estimator.score(X_test, y_test)
        print(f'Accuracy of the final model is {score}')
        return score
    
    def save_model(self, path):
        joblib.dump(self.estimator, path)
        print(f'Model successfully saved to {path}')

In [29]:
# Лучшая модель из ModelSelection
final_model = RandomForestClassifier(
    n_estimators=70,
    max_depth=20,
    criterion='entropy',
    class_weight=None,
    random_state=21
)

final = Finalize(final_model)
final.final_score(X_train, y_train, X_test, y_test)
final.save_model('../data/final_model.pkl')

Accuracy of the final model is 0.8994082840236687
Model successfully saved to ../data/final_model.pkl


In [30]:
# Загрузка сохранённой только что модели
model_load = joblib.load('../data/final_model.pkl')

lm = Finalize(model_load)
lm.final_score(X_train, y_train, X_test, y_test);
# final.save_model('../data/final_model.pkl')

Accuracy of the final model is 0.8994082840236687


In [31]:
# скор тот же - модель сохраняется корректно

In [32]:
shablon_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=22,
    criterion='entropy',
    class_weight=None,
    random_state=21
)

shablon = Finalize(shablon_model)
shablon.final_score(X_train, y_train, X_test, y_test)
# final.save_model

Accuracy of the final model is 0.9053254437869822


0.9053254437869822

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [33]:
# 1. данные подготовлены в начале ноутбука

# 2. Конвейер предобработки
preprocessing = Pipeline([
    ('feature_extractor', FeatureExtractor()),
    ('onehot_encoder', MyOneHotEncoder('dayofweek'))
])

# 3. Применение конвейера
data = preprocessing.fit_transform(df)

# data возвращает кортеж (X, y) из MyOneHotEncoder
if isinstance(data, tuple):
    X, y = data
else:
    X = data
    y = df['dayofweek']

# 4. Разбиение на train/valid/test
tvt = TrainValidationTest(test_size=0.2, valid_size=0.2, random_state=21)
X_train, X_valid, X_test, y_train, y_valid, y_test = tvt.fit_transform(X, y)

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

X_train: (1078, 43), X_test: (338, 43)


In [34]:
# 5. ModelSelection
gs_svm = GridSearchCV(estimator=SVC(), param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=-1)
gs_tree = GridSearchCV(estimator=DecisionTreeClassifier(), param_grid=tree_params, scoring='accuracy', cv=2, n_jobs=-1)
gs_rf = GridSearchCV(estimator=RandomForestClassifier(), param_grid=rf_params, scoring='accuracy', cv=2, n_jobs=-1)

grids = [gs_svm, gs_tree, gs_rf]
grid_dict = {0: 'SVM', 1: 'Decision Tree', 2: 'Random Forest'}

ms = ModelSelection(grids, grid_dict)
best_classifier = ms.choose(X_train, y_train, X_valid, y_valid)

# Датафрейм лучших результатов
results_df = ms.best_results()
print('\nBest results:')
print(results_df.to_string(index=False))

Models:   0%|          | 0/3 [00:00<?, ?it/s]


Estimator: SVM
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.952
Validation set accuracy score for best params: 0.878

Estimator: Decision Tree
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.994
Validation set accuracy score for best params: 0.867

Estimator: Random Forest
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 20, 'n_estimators': 70, 'random_state': 21}
Best training accuracy: 1.000
Validation set accuracy score for best params: 0.904

Classifier with best validation set accuracy: Random Forest

Best results:
         model                                                                                                      params  valid_score
           SVM  {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}     0.87777

In [35]:
# 6. Finalize
best_params = results_df.loc[results_df['model'] == best_classifier, 'params'].iloc[0]

if best_classifier == 'Random Forest':
    final_model = RandomForestClassifier(**best_params)
elif best_classifier == 'SVM':
    final_model = SVC(**best_params)
else:
    final_model = DecisionTreeClassifier(**best_params)

final = Finalize(final_model)
test_accuracy = final.final_score(X_train, y_train, X_test, y_test)

# Сохранение с именем файла содержащим accuracy
accuracy_str = str(round(test_accuracy, 6)).replace('.', '_')
filename = f'{best_classifier.replace(" ", "_")}_{accuracy_str}.sav'
final.save_model(f'../data/{filename}')

Accuracy of the final model is 0.8994082840236687
Model successfully saved to ../data/Random_Forest_0_899408.sav


In [36]:
moe = MyOneHotEncoder(target_column='dayofweek')
X, y = moe.fit_transform(df_extracted)

print('dayofweek в X:', 'dayofweek' in X.columns)  # False
print('dayofweek в y:', y.name if hasattr(y, 'name') else 'Series')  # dayofweek 
print('Кодированные колонки:', [c for c in X.columns if 'dayofweek' in c])  # Пусто 

dayofweek в X: False
dayofweek в y: dayofweek
Кодированные колонки: []
